In [3]:
import os
import pandas as pd
import numpy as np
from IPython.display import display

script_dir = os.getcwd() 

subjects = [f"SUB{i}" for i in range(1, 13)]

# PERBAIKAN: Menambahkan 'WER' tepat setelah 'cer' pada daftar metrik
metrics = ['cer', 'WER', 'BLEU-1', 'BLEU-2', 'BLEU-3', 'BLEU-4', 'ROUGE-1-P', 'ROUGE-1-R', 'ROUGE-1-F', 'BertScore']

print("Mengekstrak dan menghitung nilai seluruh metrik dari eksperimen Log-Mel Spectrogram...\n")

experiments = [
    {
        'Decoder': 'LSTM',
        'Train Data': 'Single',
        'type': 'single',
        'template': "complete_metrics_NOISE_BASELINE_{subject}_eq_3_0_fixed_hilbert_test_predictions_6_1.csv"
    },
    {
        'Decoder': 'IndoGPT',
        'Train Data': 'Single',
        'type': 'single',
        'template': "complete_metrics_NOISE_BASELINE_{subject}_eq_3_0_hilbert_test_predictions_10_1_IndoGPT.csv"
    },
    {
        'Decoder': 'LSTM',
        'Train Data': 'All',
        'type': 'all',
        'template': "complete_metrics_NOISE_BASELINE_all_eq_3_0_fixed_hilbert_test_predictions_1_1.csv"
    },
    {
        'Decoder': 'IndoGPT',
        'Train Data': 'All',
        'type': 'all',
        'template': "complete_metrics_NOISE_BASELINE_all_eq_3_0_hilbert_test_predictions_2_1_IndoGPT.csv"
    }
]

tables_data = {m: [] for m in metrics}

summary_rows = []


for exp in experiments:
    summary_row = {
        'Decoder': exp['Decoder'],
        'Train Data': exp['Train Data']
    }
    
    exp_metric_rows = {m: {'Decoder': exp['Decoder'], 'Train Data': exp['Train Data']} for m in metrics}
    
    raw_values = {m: [] for m in metrics}
    
    if exp['type'] == 'single':
        for subject in subjects:
            filename = exp['template'].format(subject=subject)
            filepath = os.path.join(script_dir, filename)
            
            if os.path.exists(filepath):
                try:
                    df = pd.read_csv(filepath)
                    for m in metrics:
                        if m in df.columns:
                            exp_metric_rows[m][subject] = round(df[m].mean(), 4)
                            raw_values[m].extend(df[m].dropna().tolist())
                        else:
                            exp_metric_rows[m][subject] = np.nan
                except Exception as e:
                    print(f"  [ERROR] Gagal membaca {filename}: {e}")
                    for m in metrics: exp_metric_rows[m][subject] = np.nan
            else:
                for m in metrics: exp_metric_rows[m][subject] = np.nan
                
    elif exp['type'] == 'all':
        filename = exp['template']
        filepath = os.path.join(script_dir, filename)
        
        if os.path.exists(filepath):
            try:
                df_all = pd.read_csv(filepath)
                for subject in subjects:
                    sub_df = df_all[df_all['subject'] == subject]
                    if len(sub_df) > 0:
                        for m in metrics:
                            if m in sub_df.columns:
                                exp_metric_rows[m][subject] = round(sub_df[m].mean(), 4)
                                raw_values[m].extend(sub_df[m].dropna().tolist())
                            else:
                                exp_metric_rows[m][subject] = np.nan
                    else:
                        for m in metrics: exp_metric_rows[m][subject] = np.nan
            except Exception as e:
                print(f"  [ERROR] Gagal membaca {filename}: {e}")
                for subject in subjects:
                    for m in metrics: exp_metric_rows[m][subject] = np.nan
        else:
            print(f"  ⚠️ [INFO] File {filename} belum ditemukan.")
            for subject in subjects:
                for m in metrics: exp_metric_rows[m][subject] = np.nan

    for m in metrics:
        if raw_values[m]:
            global_avg = round(np.mean(raw_values[m]), 4)
        else:
            global_avg = np.nan
            
        exp_metric_rows[m]['Rata-rata Global'] = global_avg
        tables_data[m].append(exp_metric_rows[m])
        
        summary_row[m] = global_avg
        
    summary_rows.append(summary_row)

column_order = ['Decoder', 'Train Data'] + subjects + ['Rata-rata Global']

for m in metrics:
    df_metric = pd.DataFrame(tables_data[m])
    df_metric = df_metric[[col for col in column_order if col in df_metric.columns]]
    
    print("=" * 110)
    print(f"TABEL PERBANDINGAN PERFORMA: {m.upper()}")
    print("=" * 110)
    display(df_metric)
    print("\n")

df_summary_global = pd.DataFrame(summary_rows)

summary_col_order = ['Decoder', 'Train Data'] + metrics
df_summary_global = df_summary_global[[col for col in summary_col_order if col in df_summary_global.columns]]

print("*" * 110)
print("🏆 TABEL GRAND SUMMARY: RATA-RATA GLOBAL SELURUH METRIK 🏆")
print("*" * 110)
display(df_summary_global)



CURRENT_DIR = os.getcwd()
DATASET_DIR = os.path.abspath(os.path.join(CURRENT_DIR, '../../../dataset'))

oov_summary_list = []

print("\n\n" + "=" * 110)
print("MEMULAI ANALISIS OOV KARAKTER...")
print("=" * 110)

for subject in subjects:
    train_file = os.path.join(DATASET_DIR, f"{subject}_eq_3_0_train.csv")
    val_file = os.path.join(DATASET_DIR, f"{subject}_eq_3_0_val.csv")
    test_file = os.path.join(DATASET_DIR, f"{subject}_eq_3_0_test.csv")
    
    if os.path.exists(train_file) and os.path.exists(val_file) and os.path.exists(test_file):
        df_train = pd.read_csv(train_file)
        df_val = pd.read_csv(val_file)
        df_test = pd.read_csv(test_file)
        
        # Gunakan astype(str) untuk mencegah error jika ada data null/kosong
        train_text = "".join(df_train['sentence'].astype(str).tolist())
        test_text = "".join(df_test['sentence'].astype(str).tolist())
        
        train_chars = set(train_text)
        test_chars = set(test_text)
        
        oov_chars = test_chars - train_chars
        
        test_text_len = len(test_text)
        if test_text_len > 0:
            oov_occurrences = sum(1 for char in test_text if char in oov_chars)
            oov_rate = (oov_occurrences / test_text_len) * 100
        else:
            oov_occurrences = 0
            oov_rate = 0.0

        oov_summary_list.append({
            'Subject': subject,
            'Train (Baris)': len(df_train),
            'Val (Baris)': len(df_val),
            'Test (Baris)': len(df_test),
            'Kamus Train': len(train_chars),
            'Kamus Test': len(test_chars),
            'Karakter OOV': "".join(sorted(list(oov_chars))) if oov_chars else "-",
            'Total Muncul OOV': oov_occurrences,
            'OOV Rate (%)': round(oov_rate, 4)
        })
    else:
        # Jika salah satu dari 3 file per subjek tidak ada, berikan peringatan
        print(f"  ⚠️ Melewati {subject}: File train/val/test tidak lengkap di folder dataset.")

if oov_summary_list:
    df_summary = pd.DataFrame(oov_summary_list)

    print("\n" + "=" * 110)
    print("RINGKASAN DATA SPLIT & OOV KARAKTER PER SUBJEK")
    print("=" * 110)
    display(df_summary)
else:
    print("❌ Tidak ada data yang bisa ditampilkan. Pastikan file *_eq_3_0_*.csv sudah dibuat di dalam folder /dataset.")

Mengekstrak dan menghitung nilai seluruh metrik dari eksperimen Log-Mel Spectrogram...

TABEL PERBANDINGAN PERFORMA: CER


,Decoder,Train Data,SUB1,SUB2,SUB3,SUB4,SUB5,SUB6,SUB7,SUB8,SUB9,SUB10,SUB11,SUB12,Rata-rata Global
0,LSTM,Single,77.2763,76.4113,81.6565,77.0896,75.8823,78.7763,77.3053,76.4407,79.0455,75.9374,77.6572,80.7746,77.5937
1,IndoGPT,Single,71.2098,83.1085,81.4308,85.5202,77.2880,76.2628,82.3112,68.0674,75.7801,72.3029,73.7698,71.3624,76.3958
2,LSTM,All,74.8834,73.4683,75.2256,74.8789,73.1234,74.1187,73.1098,73.4189,77.7201,73.6300,74.3684,79.4974,74.5200
3,IndoGPT,All,72.5891,73.0550,82.1308,73.2740,75.8303,77.2978,77.7727,71.1716,81.3968,70.1105,74.1882,77.3066,75.1393




TABEL PERBANDINGAN PERFORMA: WER


,Decoder,Train Data,SUB1,SUB2,SUB3,SUB4,SUB5,SUB6,SUB7,SUB8,SUB9,SUB10,SUB11,SUB12,Rata-rata Global
0,LSTM,Single,99.0357,94.3333,100.8333,100.6667,98.8889,99.6667,102.3750,101.2500,99.1228,98.9474,101.1765,100.0000,99.9141
1,IndoGPT,Single,99.7738,100.0000,107.5000,117.2500,93.5000,99.0000,100.0000,92.4524,98.9474,95.7018,99.0196,93.7500,100.1915
2,LSTM,All,99.4048,97.5000,100.0000,98.0000,96.3333,96.9167,96.3333,97.0833,99.2105,96.8797,98.8235,106.2500,98.0096
3,IndoGPT,All,101.0714,102.0000,107.5000,102.7500,100.2222,104.5000,103.5357,96.9167,110.3509,94.7368,101.1765,108.3333,102.2365




TABEL PERBANDINGAN PERFORMA: BLEU-1


...

,Decoder,Train Data,SUB1,SUB2,SUB3,SUB4,SUB5,SUB6,SUB7,SUB8,SUB9,SUB10,SUB11,SUB12,Rata-rata Global
0,LSTM,Single,45.3645,43.3954,41.6084,47.2544,43.8096,42.2652,43.8646,40.0932,44.6368,47.9444,43.8527,46.7585,44.2145
1,IndoGPT,Single,51.4446,42.9686,45.9766,48.0397,43.5865,51.8010,44.8084,49.7983,50.1139,51.3286,49.7542,54.5346,48.8599
2,LSTM,All,51.3562,52.6454,50.3696,54.7854,53.0981,50.3720,52.6836,51.6068,52.0975,51.9000,52.3259,45.1818,51.9758
3,IndoGPT,All,51.0441,49.8955,49.6792,51.9521,50.1307,48.3513,48.4110,49.0857,46.8681,53.5768,49.7631,44.1846,49.7625




**************************************************************************************************************
🏆 TABEL GRAND SUMMARY: RATA-RATA GLOBAL SELURUH METRIK 🏆
**************************************************************************************************************


,Decoder,Train Data,cer,WER,BLEU-1,BLEU-2,BLEU-3,BLEU-4,ROUGE-1-P,ROUGE-1-R,ROUGE-1-F,BertScore
0,LSTM,Single,77.5937,99.9141,2.5645,1.0162,0.8895,0.8280,6.0758,3.3197,4.1681,44.2145
1,IndoGPT,Single,76.3958,100.1915,4.3792,1.7184,1.3752,1.2913,7.2134,4.9679,5.7439,48.8599
2,LSTM,All,74.5200,98.0096,4.5037,1.7120,1.5939,1.4926,8.0423,5.1751,6.2127,51.9758
3,IndoGPT,All,75.1393,102.2365,5.4728,2.6320,2.0358,1.7954,6.8959,5.8776,6.2696,49.7625




MEMULAI ANALISIS OOV KARAKTER...

RINGKASAN DATA SPLIT & OOV KARAKTER PER SUBJEK


,Subject,Train (Baris),Val (Baris),Test (Baris),Kamus Train,Kamus Test,Karakter OOV,Total Muncul OOV,OOV Rate (%)
0,SUB1,70,10,20,24,23,-,0,0.0000
1,SUB2,35,5,10,23,23,z,1,0.3125
2,SUB3,35,5,10,23,21,-,0,0.0000
3,SUB4,70,10,20,24,22,-,0,0.0000
4,SUB5,35,5,10,23,23,-,0,0.0000
5,SUB6,70,10,20,23,22,-,0,0.0000
6,SUB7,70,10,20,24,22,-,0,0.0000
7,SUB8,70,10,20,23,23,-,0,0.0000
8,SUB9,64,9,19,24,22,-,0,0.0000
9,SUB10,63,9,19,23,23,-,0,0.0000


In [1]:
import os
import pandas as pd
import numpy as np
from IPython.display import display

script_dir = os.getcwd() 

FINAL_COMP_DIR = os.path.join(script_dir, 'final_comparison')
os.makedirs(FINAL_COMP_DIR, exist_ok=True)
print(f"📁 Folder output untuk CSV disiapkan di: {FINAL_COMP_DIR}\n")

subjects = [f"SUB{i}" for i in range(1, 13)]

# PERBAIKAN: Menambahkan 'WER' tepat setelah 'cer' pada daftar metrik
metrics = ['cer', 'WER', 'BLEU-1', 'BLEU-2', 'BLEU-3', 'BLEU-4', 'ROUGE-1-P', 'ROUGE-1-R', 'ROUGE-1-F', 'BertScore']

print("Mengekstrak dan menghitung nilai seluruh metrik dari eksperimen Log-Mel Spectrogram...\n")

experiments = [
    {
        'Decoder': 'LSTM',
        'Train Data': 'Single',
        'type': 'single',
        'template': "complete_metrics_NOISE_BASELINE_{subject}_eq_3_0_fixed_hilbert_test_predictions_6_1.csv"
    },
    {
        'Decoder': 'IndoGPT',
        'Train Data': 'Single',
        'type': 'single',
        'template': "complete_metrics_NOISE_BASELINE_{subject}_eq_3_0_hilbert_test_predictions_10_1_IndoGPT.csv"
    },
    {
        'Decoder': 'LSTM',
        'Train Data': 'All',
        'type': 'all',
        'template': "complete_metrics_NOISE_BASELINE_all_eq_3_0_fixed_hilbert_test_predictions_1_1.csv"
    },
    {
        'Decoder': 'IndoGPT',
        'Train Data': 'All',
        'type': 'all',
        'template': "complete_metrics_NOISE_BASELINE_all_eq_3_0_hilbert_test_predictions_2_1_IndoGPT.csv"
    }
]

tables_data = {m: [] for m in metrics}

summary_rows = []


for exp in experiments:
    summary_row = {
        'Decoder': exp['Decoder'],
        'Train Data': exp['Train Data']
    }
    
    exp_metric_rows = {m: {'Decoder': exp['Decoder'], 'Train Data': exp['Train Data']} for m in metrics}
    
    raw_values = {m: [] for m in metrics}
    
    if exp['type'] == 'single':
        for subject in subjects:
            filename = exp['template'].format(subject=subject)
            filepath = os.path.join(script_dir, filename)
            
            if os.path.exists(filepath):
                try:
                    df = pd.read_csv(filepath)
                    for m in metrics:
                        if m in df.columns:
                            exp_metric_rows[m][subject] = round(df[m].mean(), 4)
                            raw_values[m].extend(df[m].dropna().tolist())
                        else:
                            exp_metric_rows[m][subject] = np.nan
                except Exception as e:
                    print(f"  [ERROR] Gagal membaca {filename}: {e}")
                    for m in metrics: exp_metric_rows[m][subject] = np.nan
            else:
                for m in metrics: exp_metric_rows[m][subject] = np.nan
                
    elif exp['type'] == 'all':
        filename = exp['template']
        filepath = os.path.join(script_dir, filename)
        
        if os.path.exists(filepath):
            try:
                df_all = pd.read_csv(filepath)
                for subject in subjects:
                    sub_df = df_all[df_all['subject'] == subject]
                    if len(sub_df) > 0:
                        for m in metrics:
                            if m in sub_df.columns:
                                exp_metric_rows[m][subject] = round(sub_df[m].mean(), 4)
                                raw_values[m].extend(sub_df[m].dropna().tolist())
                            else:
                                exp_metric_rows[m][subject] = np.nan
                    else:
                        for m in metrics: exp_metric_rows[m][subject] = np.nan
            except Exception as e:
                print(f"  [ERROR] Gagal membaca {filename}: {e}")
                for subject in subjects:
                    for m in metrics: exp_metric_rows[m][subject] = np.nan
        else:
            print(f"  ⚠️ [INFO] File {filename} belum ditemukan.")
            for subject in subjects:
                for m in metrics: exp_metric_rows[m][subject] = np.nan

    for m in metrics:
        if raw_values[m]:
            global_avg = round(np.mean(raw_values[m]), 4)
        else:
            global_avg = np.nan
            
        exp_metric_rows[m]['Rata-rata Global'] = global_avg
        tables_data[m].append(exp_metric_rows[m])
        
        summary_row[m] = global_avg
        
    summary_rows.append(summary_row)

column_order = ['Decoder', 'Train Data'] + subjects + ['Rata-rata Global']

for m in metrics:
    df_metric = pd.DataFrame(tables_data[m])
    df_metric = df_metric[[col for col in column_order if col in df_metric.columns]]
    
    print("=" * 110)
    print(f"TABEL PERBANDINGAN PERFORMA: {m.upper()}")
    print("=" * 110)
    display(df_metric)
    
    metric_csv_path = os.path.join(FINAL_COMP_DIR, f"hilbert_noise_{m}.csv")
    df_metric.to_csv(metric_csv_path, index=False)
    print(f"✅ Tersimpan: {metric_csv_path}\n")

df_summary_global = pd.DataFrame(summary_rows)

summary_col_order = ['Decoder', 'Train Data'] + metrics
df_summary_global = df_summary_global[[col for col in summary_col_order if col in df_summary_global.columns]]

print("*" * 110)
print("🏆 TABEL GRAND SUMMARY: RATA-RATA GLOBAL SELURUH METRIK 🏆")
print("*" * 110)
display(df_summary_global)

summary_csv_path = os.path.join(FINAL_COMP_DIR, "hilbert_noise_grand_summary.csv")
df_summary_global.to_csv(summary_csv_path, index=False)
print(f"✅ Tersimpan: {summary_csv_path}\n")


CURRENT_DIR = os.getcwd()
DATASET_DIR = os.path.abspath(os.path.join(CURRENT_DIR, '../../../dataset'))

oov_summary_list = []

print("\n\n" + "=" * 110)
print("MEMULAI ANALISIS OOV KARAKTER...")
print("=" * 110)

for subject in subjects:
    train_file = os.path.join(DATASET_DIR, f"{subject}_eq_3_0_train.csv")
    val_file = os.path.join(DATASET_DIR, f"{subject}_eq_3_0_val.csv")
    test_file = os.path.join(DATASET_DIR, f"{subject}_eq_3_0_test.csv")
    
    if os.path.exists(train_file) and os.path.exists(val_file) and os.path.exists(test_file):
        df_train = pd.read_csv(train_file)
        df_val = pd.read_csv(val_file)
        df_test = pd.read_csv(test_file)
        
        # Gunakan astype(str) untuk mencegah error jika ada data null/kosong
        train_text = "".join(df_train['sentence'].astype(str).tolist())
        test_text = "".join(df_test['sentence'].astype(str).tolist())
        
        train_chars = set(train_text)
        test_chars = set(test_text)
        
        oov_chars = test_chars - train_chars
        
        test_text_len = len(test_text)
        if test_text_len > 0:
            oov_occurrences = sum(1 for char in test_text if char in oov_chars)
            oov_rate = (oov_occurrences / test_text_len) * 100
        else:
            oov_occurrences = 0
            oov_rate = 0.0

        oov_summary_list.append({
            'Subject': subject,
            'Train (Baris)': len(df_train),
            'Val (Baris)': len(df_val),
            'Test (Baris)': len(df_test),
            'Kamus Train': len(train_chars),
            'Kamus Test': len(test_chars),
            'Karakter OOV': "".join(sorted(list(oov_chars))) if oov_chars else "-",
            'Total Muncul OOV': oov_occurrences,
            'OOV Rate (%)': round(oov_rate, 4)
        })
    else:
        # Jika salah satu dari 3 file per subjek tidak ada, berikan peringatan
        print(f"  ⚠️ Melewati {subject}: File train/val/test tidak lengkap di folder dataset.")

if oov_summary_list:
    df_summary = pd.DataFrame(oov_summary_list)

    print("\n" + "=" * 110)
    print("RINGKASAN DATA SPLIT & OOV KARAKTER PER SUBJEK")
    print("=" * 110)
    display(df_summary)
    
    oov_csv_path = os.path.join(FINAL_COMP_DIR, "oov_tokenizer_char.csv")
    df_summary.to_csv(oov_csv_path, index=False)
    print(f"✅ Tersimpan: {oov_csv_path}\n")
else:
    print("❌ Tidak ada data yang bisa ditampilkan. Pastikan file *_eq_3_0_*.csv sudah dibuat di dalam folder /dataset.")

📁 Folder output untuk CSV disiapkan di: d:\GitRepos\EEG-to-Text_Conformer-Transducer\src\pipelines\training\final_comparison

Mengekstrak dan menghitung nilai seluruh metrik dari eksperimen Log-Mel Spectrogram...

TABEL PERBANDINGAN PERFORMA: CER


,Decoder,Train Data,SUB1,SUB2,SUB3,SUB4,SUB5,SUB6,SUB7,SUB8,SUB9,SUB10,SUB11,SUB12,Rata-rata Global
0,LSTM,Single,77.2763,76.4113,81.6565,77.0896,75.8823,78.7763,77.3053,76.4407,79.0455,75.9374,77.6572,80.7746,77.5937
1,IndoGPT,Single,71.2098,83.1085,81.4308,85.5202,77.2880,76.2628,82.3112,68.0674,75.7801,72.3029,73.7698,71.3624,76.3958
2,LSTM,All,74.8834,73.4683,75.2256,74.8789,73.1234,74.1187,73.1098,73.4189,77.7201,73.6300,74.3684,79.4974,74.5200
3,IndoGPT,All,72.5891,73.0550,82.1308,73.2740,75.8303,77.2978,77.7727,71.1716,81.3968,70.1105,74.1882,77.3066,75.1393


✅ Tersimpan: d:\GitRepos\EEG-to-Text_Conformer-Transducer\src\pipelines\training\final_comparison\hilbert_noise_cer.csv

TABEL PERBANDINGAN PERFORMA: WER


,Decoder,Train Data,SUB1,SUB2,SUB3,SUB4,SUB5,SUB6,SUB7,SUB8,SUB9,SUB10,SUB11,SUB12,Rata-rata Global
0,LSTM,Single,99.0357,94.3333,100.8333,100.6667,98.8889,99.6667,102.3750,101.2500,99.1228,98.9474,101.1765,100.0000,99.9141
1,IndoGPT,Single,99.7738,100.0000,107.5000,117.2500,93.5000,99.0000,100.0000,92.4524,98.9474,95.7018,99.0196,93.7500,100.1915
2,LSTM,All,99.4048,97.5000,100.0000,98.0000,96.3333,96.9167,96.3333,97.0833,99.2105,96.8797,98.8235,106.2500,98.0096
3,IndoGPT,All,101.0714,102.0000,107.5000,102.7500,100.2222,104.5000,103.5357,96.9167,110.3509,94.7368,101.1765,108.3333,102.2365


✅ Tersimpan: d:\GitRepos\EEG-to-Text_Conformer-Transducer\src\pipelines\training\final_comparison\hilbert_noise_WER.csv

TABEL PERBANDINGAN PERFORMA: BLEU-1


...

✅ Tersimpan: d:\GitRepos\EEG-to-Text_Conformer-Transducer\src\pipelines\training\final_comparison\hilbert_noise_BertScore.csv

**************************************************************************************************************
🏆 TABEL GRAND SUMMARY: RATA-RATA GLOBAL SELURUH METRIK 🏆
**************************************************************************************************************


,Decoder,Train Data,cer,WER,BLEU-1,BLEU-2,BLEU-3,BLEU-4,ROUGE-1-P,ROUGE-1-R,ROUGE-1-F,BertScore
0,LSTM,Single,77.5937,99.9141,2.5645,1.0162,0.8895,0.8280,6.0758,3.3197,4.1681,44.2145
1,IndoGPT,Single,76.3958,100.1915,4.3792,1.7184,1.3752,1.2913,7.2134,4.9679,5.7439,48.8599
2,LSTM,All,74.5200,98.0096,4.5037,1.7120,1.5939,1.4926,8.0423,5.1751,6.2127,51.9758
3,IndoGPT,All,75.1393,102.2365,5.4728,2.6320,2.0358,1.7954,6.8959,5.8776,6.2696,49.7625


✅ Tersimpan: d:\GitRepos\EEG-to-Text_Conformer-Transducer\src\pipelines\training\final_comparison\hilbert_noise_grand_summary.csv



MEMULAI ANALISIS OOV KARAKTER...

RINGKASAN DATA SPLIT & OOV KARAKTER PER SUBJEK


,Subject,Train (Baris),Val (Baris),Test (Baris),Kamus Train,Kamus Test,Karakter OOV,Total Muncul OOV,OOV Rate (%)
0,SUB1,70,10,20,24,23,-,0,0.0000
1,SUB2,35,5,10,23,23,z,1,0.3125
2,SUB3,35,5,10,23,21,-,0,0.0000
3,SUB4,70,10,20,24,22,-,0,0.0000
4,SUB5,35,5,10,23,23,-,0,0.0000
5,SUB6,70,10,20,23,22,-,0,0.0000
6,SUB7,70,10,20,24,22,-,0,0.0000
7,SUB8,70,10,20,23,23,-,0,0.0000
8,SUB9,64,9,19,24,22,-,0,0.0000
9,SUB10,63,9,19,23,23,-,0,0.0000


✅ Tersimpan: d:\GitRepos\EEG-to-Text_Conformer-Transducer\src\pipelines\training\final_comparison\oov_tokenizer_char.csv

